# Safe Bayesian Optimization：安全制約付きBOの実践ガイド

Safe BOでは、**目的関数を改善しながら、安全制約を満たすと推定される条件だけを提案**します。このNotebookは1次元の合成例を使い、初期安全観測 → 目的GP・安全GPの学習 → 候補提案 → 実験 → `tell()` → 再学習 → 次の候補提案を実行します。

既存の `SafeBOWorkflow`、`SafetySpecification`、`SingleTaskGP.fit()` を利用し、独自の安全獲得関数は実装しません。

**注意：** GPの信頼区間による安全判定は実機の安全を保証しません。実際の装置には独立した安全装置や運転制限が必要です。


## まず理解する3つの役割

| 要素 | 何を学習・管理するか |
|---|---|
| 目的関数GP | 品質や性能など、**最大化したい応答** |
| 安全制約GP | 温度・圧力など、**守る必要がある安全出力** |
| `SafeBOWorkflow` | 安全と予測される候補の提案、実測安全出力の登録 |

`PosteriorMean` は目的GPの予測平均を高くする獲得関数です。**安全判定そのものは安全GPと `SafetySpecification` によって扱われます**。

In [ ]:
import torch
from botorch.acquisition.analytic import PosteriorMean

from robotorchan.models.standard import SingleTaskGP
from robotorchan.optim import SafeBOWorkflow
from robotorchan.semantics import ContinuousConstraint, SafetySpecification

torch.set_default_dtype(torch.double)
bounds = torch.tensor([[0.0], [1.0]])


def objective(x):
    return -(x - 0.3).square()


def safety_output(x):
    return x - 0.7


def fit_gp(x, y):
    model = SingleTaskGP(x, y)
    model.fit()
    model.eval()
    return model


### 例題の目的関数と真の安全境界

目的関数 `f(x)=-(x-0.3)^2` は `x=0.3` で最大値0です。安全制約 `g(x)=x-0.7 <= 0` は、実際には `x<=0.7` を意味します。

ただし、**この境界は例題の作成者だけが知っている真値**です。安全GPには境界を直接与えず、観測データから推定させます。

## 1. 初期データと安全条件を準備する

ここでは入力 `x` を0〜1の範囲で動かし、目的関数 `objective(x)` を大きくしたいとします。

一方、安全出力は `g(x) = x - 0.7` とし、**`g(x) <= 0`** を守る必要があります。つまり、この例では `x <= 0.7` が実際の安全領域です（学習側にはこの境界を直接教えず、観測データから予測します）。

- `initial_x`：安全と確認した初期条件（3点）
- `initial_g`：各条件で実測した安全出力
- `ContinuousConstraint(0, 0.0)`：安全出力の0番目が0以下という条件
- `beta=4.0`：不確実性をどれほど保守的に扱うかを決める係数

`SafetySpecification` は安全条件と初期観測をまとめ、`SafeBOWorkflow` は候補提案と観測記録を担当します。

In [ ]:
initial_x = torch.tensor([[0.05], [0.10], [0.15]])
initial_g = safety_output(initial_x)
spec = SafetySpecification(
    constraints=(ContinuousConstraint(0, 0.0),),
    initial_X=initial_x,
    observed_constraints=initial_g,
    beta=4.0,
)
workflow = SafeBOWorkflow(spec, bounds)
train_x = initial_x.clone()
train_y = objective(train_x)
train_g = initial_g.clone()
print("初期安全観測:", train_x.squeeze(-1).tolist())


### 初期安全データを可視化する

最初の3点は事前に安全が確認された条件です。初期安全点を用意せずに、未知の領域を自由に探索させることはできません。

図の安全境界は**合成例の真値**であり、実際の探索時に利用できる情報ではありません。

In [ ]:
import matplotlib.pyplot as plt

plot_x = torch.linspace(0.0, 1.0, 201).unsqueeze(-1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(plot_x.squeeze(-1), objective(plot_x).squeeze(-1), label="True objective (demo only)")
axes[0].scatter(train_x.squeeze(-1), train_y.squeeze(-1), label="Initially observed")
axes[0].set(xlabel="x", ylabel="Objective", title="Objective to maximize")
axes[0].legend()
axes[1].plot(plot_x.squeeze(-1), safety_output(plot_x).squeeze(-1), label="True safety output (demo only)")
axes[1].axhline(0.0, color="gray", linestyle="--", label="Safety threshold")
axes[1].scatter(train_x.squeeze(-1), train_g.squeeze(-1), label="Initially verified safe")
axes[1].set(xlabel="x", ylabel="Safety output g(x)", title="Safe if g(x) <= 0")
axes[1].legend()
plt.show()

## 2. GPを学習して次の実験条件を提案する

目的関数と安全出力は異なるので、ここでは**2つのGP**を別々に学習します。

`SingleTaskGP` は **robotorchan版**を使用します。BoTorchの`SingleTaskGP`を継承しており、robotorchanでは`model.fit()`で学習できます。内部の予測機能はBoTorchを利用しています。

`PosteriorMean` は「予測平均が高い点を好む」BoTorchの獲得関数です。Safe BOは獲得関数を使って候補を評価する前に、GPの信頼区間に基づいて安全候補を選別します。

`num_samples=256` は安全性を調べる候補数、`seed=10` は候補生成の再現性を保つための乱数シードです。安全な候補が見つからないときは例外になります。

In [ ]:
objective_model = fit_gp(train_x, train_y)
safety_model = fit_gp(train_x, train_g)
acquisition = PosteriorMean(objective_model)
candidate = workflow.ask(
    safety_model,
    mode="objective",
    acq_function=acquisition,
    num_samples=256,
    seed=10,
)
print("提案候補:", candidate.item())


### 目的GPと安全GPの予測を可視化する

安全判定には、**安全GPの予測平均だけでなく不確実性**が関係します。`beta` は不確実性をどの程度保守的に扱うかの係数です。

以下の帯はGPの予測平均±2標準偏差を示す**説明用の可視化**です。`SafeBOWorkflow` 内部の厳密な候補判定を再実装したものではありません。

In [ ]:
with torch.no_grad():
    objective_post = objective_model.posterior(plot_x)
    safety_post = safety_model.posterior(plot_x)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, posterior, name in ((axes[0], objective_post, "Objective GP"), (axes[1], safety_post, "Safety GP")):
    mean = posterior.mean.squeeze(-1)
    sd = posterior.variance.clamp_min(0).sqrt().squeeze(-1)
    ax.plot(plot_x.squeeze(-1), mean, label="GP mean")
    ax.fill_between(plot_x.squeeze(-1), mean-2*sd, mean+2*sd, alpha=0.2, label="Mean ± 2 SD")
    ax.axvline(candidate.item(), linestyle=":", label="Proposed candidate")
    ax.set(xlabel="x", title=name)
    ax.legend()
axes[1].axhline(0.0, linestyle="--", color="gray", label="Safety threshold")
plt.show()

## 3. 実験結果を登録する（tell）

`ask()` は「次にどこを測るか」を決めるだけです。**実験の実行と結果の取得は利用者の役割**です。

ここでは実験を関数で模擬し、`observed_y`（目的関数の観測値）と`observed_g`（安全出力の観測値）を得ます。

`workflow.tell(candidate, observed_g)` は**安全出力の実測値**を登録します。目的関数の観測値は別途`train_y`に追加し、次回のGP学習に使用します。

In [ ]:
observed_y = objective(candidate)
observed_g = safety_output(candidate)
workflow.tell(candidate, observed_g)
train_x = torch.cat([train_x, candidate], dim=0)
train_y = torch.cat([train_y, observed_y], dim=0)
train_g = torch.cat([train_g, observed_g], dim=0)
print("実測安全制約:", observed_g.item())
print("記録した観測数:", workflow.observed_X.shape[0])


### `ask()` と `tell()` の役割を区別する

`ask()` は**実験前の候補提案**です。提案しただけでは安全出力を観測していません。

`workflow.tell(candidate, observed_g)` には実測した**安全出力**を登録します。一方、目的関数の観測値は `train_y` に追加します。GPは `tell()` だけでは自動で再学習されないため、次の候補提案前に最新データで `fit()` を呼びます。

## 4. GPを更新して安全領域を探索する

新しい実測値が増えたら、`train_x`と`train_g`を使って安全GPを再学習します。**`tell()`だけではGPは自動更新されません。**

`mode="expansion"` は、現在安全と予測される点の中で、安全出力の予測不確実性が高い候補を探します。目的関数を直接最大化する`objective`モードとは役割が異なります。

この探索方法はSafeOptの厳密な「安全領域を拡大できる点」の判定ではありません。

### 安全領域の探索モード

`mode="objective"` は安全と予測される候補の中で目的関数の改善を重視します。`mode="expansion"` は安全出力の予測不確実性が大きい場所を探します。

**重要：** `expansion` は厳密なSafeOptの安全集合拡大判定と同義ではありません。また、探索候補も実測値が得られてから状態へ登録します。

In [ ]:
updated_safety_model = fit_gp(train_x, train_g)
expansion_candidate = workflow.ask(
    updated_safety_model,
    mode="expansion",
    num_samples=256,
    seed=11,
)
print("探索候補:", expansion_candidate.item())
workflow.tell(expansion_candidate, safety_output(expansion_candidate))


### 2回目の提案も実測し、GPを更新する

最初の目的重視候補と、次の安全探索候補の両方を実測した後、目的GPと安全GPを再学習します。これにより**候補提案 → 実測 → 更新 → 次の提案**の流れを一通り確認できます。

In [ ]:
expansion_y = objective(expansion_candidate)
expansion_g = safety_output(expansion_candidate)
train_x = torch.cat([train_x, expansion_candidate], dim=0)
train_y = torch.cat([train_y, expansion_y], dim=0)
train_g = torch.cat([train_g, expansion_g], dim=0)
refitted_objective_model = fit_gp(train_x, train_y)
refitted_safety_model = fit_gp(train_x, train_g)
next_objective_candidate = workflow.ask(
    refitted_safety_model,
    mode="objective",
    acq_function=PosteriorMean(refitted_objective_model),
    num_samples=256,
    seed=12,
)
print("First objective candidate:", candidate.item())
print("Expansion candidate:", expansion_candidate.item())
print("Next objective candidate:", next_objective_candidate.item())
print("Completed observations:", train_x.shape[0])

### 実測済みの候補と次の提案を比較する

実測済みの観測点と、まだ測定していない次の提案点を分けて表示します。真の目的関数・安全境界は**合成例でのみ既知**です。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(plot_x.squeeze(-1), objective(plot_x).squeeze(-1), label="True objective (demo only)")
axes[0].scatter(train_x.squeeze(-1), train_y.squeeze(-1), label="Completed observations")
axes[0].axvline(next_objective_candidate.item(), linestyle=":", label="Next proposal")
axes[0].set(xlabel="x", ylabel="Objective", title="Safe BO observations")
axes[0].legend()
axes[1].plot(plot_x.squeeze(-1), safety_output(plot_x).squeeze(-1), label="True safety output (demo only)")
axes[1].scatter(train_x.squeeze(-1), train_g.squeeze(-1), label="Completed observations")
axes[1].axhline(0.0, linestyle="--", color="gray", label="Safety threshold")
axes[1].axvline(next_objective_candidate.item(), linestyle=":", label="Next proposal")
axes[1].set(xlabel="x", ylabel="Safety output", title="Safety observations")
axes[1].legend()
plt.show()

## 5. 振り返りと実運用での注意

- `SingleTaskGP(...).fit()` でモデルを学習し、`ask()`で候補を提案し、実測後に`tell()`で安全出力を記録します。
- `beta` を大きくすると通常は保守的になりますが、確率的な安全保証を自動的に得られるわけではありません。
- 初期データは安全が確認された条件を使用してください。実測ノイズやモデルの誤差にも注意が必要です。
- 実機では、独立した安全機構や運転上の制限も必要に応じて併用してください。
- 連続・整数・カテゴリ変数を組み合わせた複数候補の提案は、`51_safe_bo_mixed_q_batch.ipynb`で説明します。

詳しい数式と安全保証の限界は、Safe BOの理論ドキュメントを参照してください。

## 6. 安全制約付きBOを実務で使う前に

| 目的 | 既存API | 注意点 |
|---|---|---|
| 目的関数と安全出力の学習 | `SingleTaskGP.fit()` | それぞれ実測値を使う |
| 安全制約の定義 | `ContinuousConstraint` | `g(x) <= 0` の符号を確認 |
| 安全仕様の管理 | `SafetySpecification` | 初期安全データと `beta` |
| 安全候補の提案 | `SafeBOWorkflow.ask()` | GP予測に基づく判断 |
| 安全実測値の登録 | `SafeBOWorkflow.tell()` | 自動GP再学習ではない |
| 目的重視の候補選択 | `PosteriorMean` | 予測平均最大化 |
| 安全領域の探索 | `mode="expansion"` | 厳密なSafeOptとは異なる |

**実務上の重要事項：** 初期点の安全確認、実測ノイズ、モデルの不確実性、安全装置・インターロック、停止条件を別途設計してください。Safe BOの候補選択だけで実機の安全を保証することはできません。

Mixed変数・q-batchの安全制約付き提案は [Notebook 51](51_safe_bo_mixed_q_batch.ipynb) を参照してください。